# Inventory planning Challenge

One decision: replenish one item with a fixed batch when inventory position reaches its trigger. Use the same functions on a new item and choose the function and arguments yourself. Estimated working time 40–50 minutes, mainly independently after class.

The data are constructed for teaching.

The same regional spare-parts distributor stocks a different replacement filter. Apply the familiar rule to this new item.


The five experiments are incoming supply, delivery time, buffer, smaller batch and larger batch.


## Terms used in this case

| Term | Meaning |
|---|---|
| Replenishment | Restocking an item. |
| Stock keeping unit (SKU) | An identifier for one item. |
| Policy | A rule specifying when and how much to order. |
| Batch Q | Pieces placed in one order. |
| Annual demand D | 365 times the earlier daily mean; a planning estimate, not a verified forecast. |
| Ordering cost K | Fixed cost per order: EUR 50 in this case. EUR means euros. |
| Holding cost h | Cost of keeping one physical piece for a year: EUR 2 here. |
| Economic order quantity (EOQ) | Batch that minimises annual ordering plus holding cost under steady demand, full deliveries, no shortages and unchanged price. |
| Lead time L | Calendar days from order date to receipt date, 4 in the baseline; compared with 7 in guided work and 6 in the Challenge. A receipt is arriving supply. |
| Mean demand | Arithmetic average of requested pieces per day. |
| Sample standard deviation | Spread of daily demand around its mean, estimated from the earlier records. |
| Safety stock / buffer | Extra protection against demand above the average during delivery time. |
| Buffer factor z | Multiplier selecting how much estimated demand spread to cover. Compare 0 with 1.65. |
| On-hand stock | Usable physical pieces at the location. |
| Incoming stock / pipeline | Confirmed pieces ordered but not yet received. |
| Backlog | Customer requests still waiting. |
| Inventory position | On hand + incoming − backlog, in pieces. |
| Reorder point r | The inventory-position trigger for placing a batch order. |
| Baseline | The reference rule for comparisons: EOQ batch and z=1.65 here. |
| Day-by-day test | Apply the rule to supplied later demand; the program calculates orders, receipts and stock. `replay` is the Python name for this later demand table. |
| Immediate fill | Requested pieces served immediately ÷ requested pieces, expressed as a percentage. |
| Shortage day | A day with some new requested pieces left waiting. |
| Closing stock | Physical stock after the day's receipts and requests. |


## Data and business conditions

One row is one calendar day: `Date` identifies the date, `SKU` identifies the item and `Demand` is requested pieces, including requests that may wait. The new item is SKU-INV-CH-002. First 90 days,1 January–31 March 2026, estimate demand; later 60 days,1 April–30 May, test the rule. Mean 25.00 pieces/day, standard deviation 3.34 pieces. Use the same illustrative K=EUR 50/order, h=EUR 2/piece/year and L=4. Calculate this item’s Q and r from its own history.
The earlier records estimate the rule; the later records supply the requests for its day-by-day test. The program calculates stock and orders rather than reading them as observed transactions.


## One rule: choose the batch, then choose its trigger

**How much:** D=365×historical mean. EOQ=√(2×D×K/h). Choose Q as the nearest whole piece. For the EOQ expression, √ means square root. With steady demand and full replenishment, average stock attributable to the batch is Q/2; annual ordering cost is (D/Q)×K and annual holding cost for that stock is (Q/2)×h. Smaller batches repeat the fixed ordering cost more often; larger batches hold more stock. This annual calculation excludes the uncertainty buffer.

**When:** expected demand during delivery=mean×L. Safety stock=z×daily standard deviation×√L. Reorder point r=expected delivery-time demand+safety stock, rounded **up** to whole pieces. Round the sum using full precision.

**Act:** monitor inventory position. When it reaches r or below, order **Q pieces**. If confirmed incoming supply already lifts position above r, wait. Each new order raises position immediately but becomes physical stock only on receipt. There is no weekly order schedule and no order-up-to target in this case.

The buffer assumes that daily demand keeps a similar average and spread and one day's demand does not predict the next. Then spread over L days grows with √L. If total delivery-time demand has a roughly bell-shaped pattern, z=1.65 gives about 95% modelled coverage. This is not a guarantee of future service or a promise that 95% of pieces are served immediately. The factor 1.65 is an illustrative chosen protection level, not estimated from the demand records or selected by a cost optimisation. Lead time is fixed within each run; Experiment 2 compares known values, not unexpected delivery delays.


## Read cost and service together

For the 60 test days, ordering cost=positive orders×EUR 50. Holding cost=sum of daily closing physical stock×EUR 2/365. This uses closing stock as a daily approximation. **Combined measured cost** adds these two 60-day costs; it is not an annual amount. Purchase cost, holding of incoming supply and monetary shortage costs are excluded. Keep immediate fill, shortage days and ending stock/backlog/incoming supply visible. Identical opening stock does not make different ending states economically equivalent. One finite replay cannot prove the lowest long-run total cost.

The annual EOQ calculation excludes the extra buffer. The 60-day test costs the actual physical stock, including the buffer’s effect. These amounts have different time horizons and must not be compared as though they covered the same period.


## Day-by-day test: timing and limits

Start with 220 physical pieces, no incoming supply and no backlog. Each day, receive supply due and serve old backlog first. Process that day's requested pieces one by one; after each request, monitor position and place Q immediately when it reaches r. Outstanding orders are included, preventing duplicate replenishment for the same trigger. A placed order arrives at the start of order date+L calendar days (L=4 in the baseline). Full supply is unlimited; unfilled requests wait, with no cancellations. Estimates are never updated using later demand.

The data contain daily totals, not times within a day. Sequential unit requests and morning receipt dates are explicit timing assumptions, not recovered historical transactions. This illustrates a continuously monitored position rule at daily data resolution; actual receipt/request timing needs operational records before adoption.


## Code and functions before use

A **function** is reusable instructions; a **parameter** is a named input and an **argument** is the value supplied to it. A return is a result saved in a variable. A **default** is the value used when an optional argument is omitted. pandas is a Python library for tables; `pd` is its short name. A **DataFrame** is a row/column table; a **Series** is one labelled set of values. `display` shows results; `pd.read_csv` reads a CSV table; `.head()` shows its first rows; `.round(2)` changes displayed decimal places. `assert` checks a condition. `history` is the earlier demand table and `replay` is the later demand table. A CSV is a comma-separated-values table. Run Setup unchanged.

| Function | Inputs / purpose | Results |
|---|---|---|
| `policy_levels(history)` | Earlier records; calculates demand, EOQ batch and reorder point using L=4,z=1.65. | Named settings. |
| `simulate(history, replay)` | Calculates stock and orders from the fixed-batch trigger rule. Optional `lead_days` sets known delivery time in calendar days; `z` changes the buffer factor; `batch_factor` multiplies the baseline whole-piece batch before rounding the changed Q. | Daily records, order records, settings, in that order. |
| `validate(daily, orders)` | Checks stock, demand, backlog, incoming supply, fixed batches and trigger decisions. | True if all checks pass; otherwise an error. |
| `metrics(daily, orders)` | Summarises cost, service and ending states. | The named measures defined below. |
| `decision_task(history, on_order)` | Separate decision example before allocating stock to waiting requests: on-hand80, backlog20; choose incoming pieces. Uses this item's Q and r. | Position, trigger, batch and new order. |
| `compare_policy(history, replay, changed, parameter)` | Compare one setting on identical demand. `parameter` is `'lead_days'`, `'z'` or `'batch_factor'`; `changed` supplies its new value. Other settings reset. For z the before comparison is 0; for delivery time it is 4; for batch size it is 1.0. | Settings, metrics, order dates and stock/backlog plots and an order-timing panel; returns changed daily records, orders and comparison table. |

`batch_factor=0.5` means half the EOQ batch, rounded to whole pieces. It is an intentional alternative for comparison; it does not change costs K/h or recalculate an alleged new optimum. In plots, **Before comparison** is the first rule and **Changed setting** the second.

### How to read the comparison plots

The top panel shows **physical stock at each daily close**. Blue open circles show the before rule; orange crosses show the changed rule. Both symbols remain visible when values coincide. These are daily snapshots: they do not show the stock path within a day. The legend gives each rule's Q (batch), r (position trigger), L (delivery days) and z (buffer factor). The trigger applies to inventory position, so it is not drawn as a line on physical stock.

The middle panel shows **requested pieces still waiting at each daily close**, with whole-piece counts. If both runs stay at zero, the panel says so.

The bottom panel shows **when each order is placed and received**. A triangle marks placement, a square marks receipt, and the segment between them is the delivery wait. Each row is one rule. This explains how an earlier order can produce the same receipt date and the same stock values when known delivery time increases. Check the order table for exact dates and quantities. Equal closing stock does not prove the rules or their order dates are identical.


## Record fields and reported measures

| Fields | Meaning, in pieces unless stated otherwise |
|---|---|
| `Date`, `SKU`, `Demand` | Calendar date, item identifier, requested pieces. |
| `Opening_On_Hand`, `Opening_Backorder` | Stock and waiting demand before morning receipts. |
| `Receipt_Qty`, `Backlog_Shipped_Qty` | Supply received and pieces used to clear old waiting demand. |
| `Immediate_Served_Qty`, `New_Unfilled_Qty` | Today's requested pieces served immediately / left waiting. |
| `Shipped_Qty` | All pieces served, including earlier waiting demand. |
| `On_Hand_Inventory`, `Backorder_Qty`, `On_Order_Qty` | Closing stock, waiting demand, incoming supply. |
| `Inventory_Position` | Closing on-hand+incoming−waiting demand. |
| `Order_Qty`, `Batch_Q`, `Reorder_Point` | Pieces newly ordered today, fixed pieces per order, trigger r. |
| Order records: `Order_Date`, `Receipt_Date`, `Position_At_Order`, `Lead_Time_Days` | Placement/arrival dates, position immediately before the batch is placed, calendar days to receipt. `Order_Qty` and `Reorder_Point` repeat their meanings. |

**Metrics:** demand pieces sum requests; immediate fill divides immediately served pieces by all requested pieces. Shortage days count days with new unfilled requests. Average closing stock averages the daily physical stock values. Orders placed counts batches; pieces ordered sums quantities. Ordering, holding and combined measured costs follow the formulas above. Closing stock, backlog and incoming pieces are the final day's states. A zero shelf balance alone does not prove a customer waited.

In the displayed order table, `Comparison` labels the before rule and changed rule so order and receipt dates can be compared directly.


In [ ]:
import warnings
warnings.filterwarnings('ignore', message="The 'generic' unit for NumPy timedelta is deprecated.*", category=DeprecationWarning)
get_ipython().run_line_magic('matplotlib', 'inline')
HELPER_SOURCE = '"""One item: fixed order quantity and inventory-position trigger. Pieces/calendar days."""\nimport math\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom IPython.display import display\n\ndef policy_levels(history, lead_days=4, z=1.65, batch_factor=1.0):\n    mean=float(history.Demand.mean());std=float(history.Demand.std(ddof=1))\n    annual=365*mean;eoq=math.sqrt(2*annual*50/2)\n    return {\'Mean demand pieces/day\':mean,\'Daily standard deviation pieces\':std,\n        \'Annual demand estimate pieces\':annual,\'Ordering cost EUR/order\':50,\n        \'Holding cost EUR/piece/year\':2,\'EOQ pieces\':eoq,\n        \'Batch Q pieces\':max(1,round(round(eoq)*batch_factor)),\'Lead time days\':lead_days,\n        \'Buffer factor z\':z,\'Expected lead-time demand pieces\':mean*lead_days,\n        \'Safety stock pieces\':z*std*math.sqrt(lead_days),\n        \'Reorder point r pieces\':math.ceil(mean*lead_days+z*std*math.sqrt(lead_days))}\n\ndef decision_task(history, on_order=0):\n    levels=policy_levels(history);ip=80+on_order-20\n    out=pd.DataFrame([{\'On hand pieces\':80,\'Incoming pieces\':on_order,\'Backlog pieces\':20,\n        \'Inventory position pieces\':ip,\'Reorder point r pieces\':levels[\'Reorder point r pieces\'],\n        \'Batch Q pieces\':levels[\'Batch Q pieces\'],\n        \'New order pieces\':levels[\'Batch Q pieces\'] if ip<=levels[\'Reorder point r pieces\'] else 0}])\n    display(out.T);return out\n\ndef simulate(history,replay,lead_days=4,z=1.65,batch_factor=1.0):\n    """Process today\'s unit requests sequentially; trigger immediately after a crossing.\n\n    Receipts occur at the start of order date+L calendar days. Earlier backlog is\n    served first. Day-level inputs have no intraday timestamps: morning receipts\n    are an explicit timing approximation. No forecast updates/lost sales.\n    Initial physical stock220, no incoming stock/backlog. Full unlimited supply.\n    """\n    if not isinstance(lead_days,(int,np.integer)) or lead_days<1:raise ValueError(\'lead_days must be a positive integer\')\n    if z<0 or not np.isfinite(z) or batch_factor<=0 or not np.isfinite(batch_factor):raise ValueError(\'Invalid buffer/batch factor\')\n    levels=policy_levels(history,lead_days,z,batch_factor);Q=levels[\'Batch Q pieces\'];r=levels[\'Reorder point r pieces\']\n    hand,backlog=220,0;pipeline=[];records=[];orders=[]\n    for t,(_,row) in enumerate(replay.iterrows()):\n        opening,old_backlog=hand,backlog\n        received=sum(q for due,q in pipeline if due==t);pipeline=[(due,q)for due,q in pipeline if due!=t]\n        hand+=received;cleared=min(hand,backlog);hand-=cleared;backlog-=cleared\n        ordered=0;trigger_positions=[]\n        def place_if_due():\n            nonlocal ordered\n            ip=hand+sum(q for _,q in pipeline)-backlog\n            while ip<=r:\n                trigger_positions.append(ip);pipeline.append((t+lead_days,Q));ordered+=Q\n                orders.append({\'Order_Date\':row.Date,\'Receipt_Date\':row.Date+pd.Timedelta(days=lead_days),\n                    \'Order_Qty\':Q,\'Position_At_Order\':ip,\'Reorder_Point\':r,\'Lead_Time_Days\':lead_days})\n                ip+=Q\n        place_if_due();served=0;unfilled=0\n        for unit in range(int(row.Demand)):\n            if hand>0:hand-=1;served+=1\n            else:backlog+=1;unfilled+=1\n            place_if_due()\n        records.append({\'Date\':row.Date,\'SKU\':row.SKU,\'Demand\':int(row.Demand),\n            \'Opening_On_Hand\':opening,\'Opening_Backorder\':old_backlog,\'Receipt_Qty\':received,\n            \'Backlog_Shipped_Qty\':cleared,\'Immediate_Served_Qty\':served,\'New_Unfilled_Qty\':unfilled,\n            \'Shipped_Qty\':cleared+served,\'On_Hand_Inventory\':hand,\'Backorder_Qty\':backlog,\n            \'On_Order_Qty\':sum(q for _,q in pipeline),\'Inventory_Position\':hand+sum(q for _,q in pipeline)-backlog,\n            \'Order_Qty\':ordered,\'Batch_Q\':Q,\'Reorder_Point\':r})\n    return pd.DataFrame(records),pd.DataFrame(orders,columns=[\'Order_Date\',\'Receipt_Date\',\'Order_Qty\',\'Position_At_Order\',\'Reorder_Point\',\'Lead_Time_Days\']),levels\n\ndef metrics(d,orders):\n    return {\'Demand pieces\':int(d.Demand.sum()),\'Immediate fill %\':100*d.Immediate_Served_Qty.sum()/d.Demand.sum(),\n        \'Shortage days\':int(d.New_Unfilled_Qty.gt(0).sum()),\'Average closing stock pieces\':float(d.On_Hand_Inventory.mean()),\n        \'Orders placed\':len(orders),\'Pieces ordered\':int(d.Order_Qty.sum()),\n        \'Ordering cost EUR\':len(orders)*50.0,\'Holding cost EUR\':float(d.On_Hand_Inventory.sum())*2/365,\n        \'Combined measured cost EUR\':len(orders)*50.0+float(d.On_Hand_Inventory.sum())*2/365,\n        \'Closing stock pieces\':int(d.On_Hand_Inventory.iloc[-1]),\'Closing backlog pieces\':int(d.Backorder_Qty.iloc[-1]),\n        \'Closing incoming pieces\':int(d.On_Order_Qty.iloc[-1])}\n\ndef compare_policy(history,replay,changed,parameter):\n    if parameter not in [\'z\',\'batch_factor\',\'lead_days\']:raise ValueError(\'Choose z, batch_factor or lead_days\')\n    before={\'z\':0.0}if parameter==\'z\'else{}\n    d0,o0,l0=simulate(history,replay,**before);d,o,l=simulate(history,replay,**{parameter:changed})\n    labels=[\'Before comparison\',\'Changed setting\']\n    display(pd.DataFrame([l0,l],index=labels)[[\'Batch Q pieces\',\'Lead time days\',\'Reorder point r pieces\',\'Safety stock pieces\']].T.round(2))\n    out=pd.DataFrame([metrics(d0,o0),metrics(d,o)],index=labels);display(out.T.round(2))\n    display(pd.concat([o0.assign(Comparison=labels[0]),o.assign(Comparison=labels[1])],ignore_index=True))\n    # Only dated closing snapshots are observed: no invented within-day stock path.\n    from matplotlib.ticker import MaxNLocator\n    import matplotlib.dates as mdates\n    fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True,\n        gridspec_kw={\'height_ratios\':[2.4,1.1,1.3]},layout=\'constrained\')\n    styles=[dict(color=\'#1763A6\',marker=\'o\',s=27,facecolors=\'none\',linewidths=1.2),\n            dict(color=\'#D35400\',marker=\'x\',s=18,linewidths=1.1)]\n    for daily,levels,label,style in [(d0,l0,labels[0],styles[0]),(d,l,labels[1],styles[1])]:\n        detail=(f"{label}: Q={levels[\'Batch Q pieces\']}, "\n                f"r={levels[\'Reorder point r pieces\']}, L={levels[\'Lead time days\']}, "\n                f"z={levels[\'Buffer factor z\']:.2f}")\n        axes[0].scatter(daily.Date,daily.On_Hand_Inventory,label=detail,**style)\n        axes[1].scatter(daily.Date,daily.Backorder_Qty,**style)\n    axes[0].set(title=\'Physical stock at the close of each day\',ylabel=\'Stock (pieces)\',ylim=(0,max(d0.On_Hand_Inventory.max(),d.On_Hand_Inventory.max())*1.35))\n    axes[0].legend(loc=\'upper left\',fontsize=8)\n    if d0.On_Hand_Inventory.equals(d.On_Hand_Inventory):\n        axes[0].text(.99,.96,\'Identical closing-stock values; both symbols shown\',\n            transform=axes[0].transAxes,ha=\'right\',va=\'top\',fontsize=8)\n    axes[0].margins(y=.25)\n    max_wait=int(max(d0.Backorder_Qty.max(),d.Backorder_Qty.max()))\n    axes[1].set(title=\'Requested pieces still waiting at the close of each day\',\n        ylabel=\'Waiting (pieces)\',ylim=(-.15,max(1,max_wait)*1.3))\n    if max_wait==0:\n        axes[1].text(.5,.6,\'No waiting demand at any daily close in either run\',\n            transform=axes[1].transAxes,ha=\'center\',fontsize=9)\n    # Each horizontal segment is an actual order\'s delivery wait, not a stock path.\n    for y,orders,style in [(1,o0,styles[0]),(0,o,styles[1])]:\n        for order in orders.itertuples():\n            axes[2].plot([order.Order_Date,order.Receipt_Date],[y,y],\n                color=style[\'color\'],linewidth=2)\n        axes[2].scatter(orders.Order_Date,[y]*len(orders),marker=\'v\',s=55,\n            color=style[\'color\'],zorder=3)\n        axes[2].scatter(orders.Receipt_Date,[y]*len(orders),marker=\'s\',s=42,\n            facecolors=\'white\',edgecolors=style[\'color\'],linewidths=1.5,zorder=4)\n    axes[2].set(title=\'Order timing: triangle = order placed; square = supply received\',\n        yticks=[0,1],yticklabels=[\'Changed setting\',\'Before comparison\'],\n        ylim=(-.6,1.6),xlabel=\'Calendar date\')\n    for ax in axes:\n        ax.grid(axis=\'x\',alpha=.18)\n    for ax in axes[:2]:\n        ax.yaxis.set_major_locator(MaxNLocator(integer=True,nbins=5))\n    locator=mdates.AutoDateLocator(minticks=5,maxticks=9)\n    axes[2].xaxis.set_major_locator(locator)\n    axes[2].xaxis.set_major_formatter(mdates.DateFormatter(\'%d %b\'))\n    fig.suptitle(\'Same demand, one policy setting changed\',fontsize=13)\n    plt.show();return d,o,out\n\ndef validate(d,orders):\n    assert d.Date.is_monotonic_increasing and not d.duplicated([\'Date\',\'SKU\']).any()\n    assert d.select_dtypes(\'number\').ge(0).all().all()\n    assert (d.Opening_On_Hand+d.Receipt_Qty-d.Shipped_Qty==d.On_Hand_Inventory).all()\n    assert (d.Opening_Backorder+d.New_Unfilled_Qty-d.Backlog_Shipped_Qty==d.Backorder_Qty).all()\n    assert (d.Immediate_Served_Qty+d.New_Unfilled_Qty==d.Demand).all()\n    assert (d.On_Hand_Inventory+d.On_Order_Qty-d.Backorder_Qty==d.Inventory_Position).all()\n    assert ((d.On_Hand_Inventory==0)|(d.Backorder_Qty==0)).all()\n    assert (d.Opening_On_Hand.iloc[1:].to_numpy()==d.On_Hand_Inventory.iloc[:-1].to_numpy()).all()\n    assert (d.Opening_Backorder.iloc[1:].to_numpy()==d.Backorder_Qty.iloc[:-1].to_numpy()).all()\n    assert int(d.Receipt_Qty.sum()+d.On_Order_Qty.iloc[-1])==int(d.Order_Qty.sum())\n    assert int(d.On_Hand_Inventory.iloc[-1]-d.Backorder_Qty.iloc[-1])==220+int(d.Receipt_Qty.sum()-d.Demand.sum())\n    assert orders.Order_Qty.eq(int(d.Batch_Q.iloc[0])).all()\n    assert orders.Position_At_Order.le(orders.Reorder_Point).all()\n    for _,o in orders.iterrows():assert(o.Receipt_Date-o.Order_Date).days==o.Lead_Time_Days\n    for _,row in d.iterrows():assert row.Receipt_Qty==orders.loc[orders.Receipt_Date.eq(row.Date),\'Order_Qty\'].sum()\n    return True\n'
exec(HELPER_SOURCE)
from pathlib import Path
source = Path('data/inventory_challenge_demand.csv')
data = pd.read_csv(source if source.exists() else 'https://raw.githubusercontent.com/saikisri97/17_Hof_Lecture_Code_Pingo/refs/heads/main/SCA_2026/Inventory_Planning/data/inventory_challenge_demand.csv', parse_dates=['Date'])
history, replay = data.iloc[:90].copy(), data.iloc[90:].copy()
assert len(history)==90 and len(replay)==60
assert not history[['Date','SKU']].duplicated().any()
assert history.Date.max() < replay.Date.min()
display(history.head(3))
display(pd.Series(policy_levels(history)).round(2))


## Experiment 1: Incoming supply

“The shelf has not changed, so another batch is still needed.” Compare 0 with 150 incoming pieces in the decision example.

**1a. Which function and arguments test the statement? Report position, Q,r and the order in each run.**

**1b. Does the statement hold? Explain how confirmed incoming supply affects position.**

**1c. What receipt evidence is needed before accepting the wait decision?**

**Function choice and reason:**

_Write your answer here._

In [ ]:
# Write your selected function call(s) here.

**Interpretation:** answer a, b and c using the displayed evidence.

_Write your answers here._

## Experiment 2: Longer delivery

“A longer delivery time means customers must wait longer.” Compare L=4 with L=6, recalculating the trigger for each.

**2a. Which function and arguments compare the rules? Report Q, r and the first order/receipt dates.**

**2b. What happens to service, stock and measured cost? Explain the role of earlier ordering.**

**2c. What assumptions limit this result?**

**Function choice and reason:**

_Write your answer here._

In [ ]:
# Write your selected function call(s) here.

**Interpretation:** answer a, b and c using the displayed evidence.

_Write your answers here._

## Experiment 3: Buffer

“A buffer cannot improve service if the batch size stays the same.” Compare z=0 with 1.65.

**3a. Which function and arguments compare the rules? Report Q,r,fill,shortage days and average stock.**

**3b. What changes in measured cost and what service benefit is observed?**

**3c. Does the result prove future service or a lowest total-cost choice?**

**Function choice and reason:**

_Write your answer here._

In [ ]:
# Write your selected function call(s) here.

**Interpretation:** answer a, b and c using the displayed evidence.

_Write your answers here._

## Experiment 4: Smaller batch

“Halving the batch cuts stock and therefore cuts cost.” Compare batch_factor=1.0 with 0.5.

**4a. Which function and arguments compare the rules? Report Q,r,average stock,orders and immediate fill.**

**4b. Report ordering,holding and combined measured costs. Does the claim hold?**

**4c. How do the ending states limit the recommendation?**

**Function choice and reason:**

_Write your answer here._

In [ ]:
# Write your selected function call(s) here.

**Interpretation:** answer a, b and c using the displayed evidence.

_Write your answers here._

## Experiment 5: Larger batch

“Doubling the batch saves ordering costs and therefore lowers combined cost.” Compare batch_factor=1.0 with 2.0.

**5a. Which function and arguments compare the rules? Report Q, r, average stock, orders and fill.**

**5b. Report both cost components and their sum. Does the statement hold?**

**5c. What do the ending states imply for a recommendation?**

**Function choice and reason:**

_Write your answer here._

In [ ]:
# Write your selected function call(s) here.

**Interpretation:** answer a, b and c using the displayed evidence.

_Write your answers here._

## Recommendation

Explain the fixed-batch trigger rule. Compare the EOQ batch with the smaller and larger batches using service, both cost components and ending commitments. Explain what the lead-time and buffer experiments add to your recommendation. State the further evidence needed. Include a comparison table and a 100–150-word recommendation.

_Add your comparison table and recommendation here._